# Diabetes 130-US Hospitals: EDA and Machine Learning

Exploratory analysis in SQL and Python, preprocessing, and three modelling tasks on 101,766 hospital encounters of diabetic patients:

- **Regression:** predict length of stay (`time_in_hospital`).
- **Classification:** predict whether diabetes medication is prescribed (`diabetesMed`).
- **Clustering:** group similar patients.

The notebook ends with feature selection, hyperparameter tuning with GridSearchCV and nested cross-validation, and model selection.

## Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1 Exploratory Data Analysis (EDA) with Python and SQL



### 1A Exploratory Data Analysis with SQL

### 1A.1 General Overview of the Data

In [ ]:
# Count the total number of records in the encounter table
"""
SELECT COUNT(*) AS total_record_number 
FROM encounter;

"""

# Check the distribution of different admission types
"""
SELECT admission_type_id, COUNT(*) AS number_of_patients
FROM encounter
GROUP BY admission_type_id
ORDER BY number_of_patients DESC;

"""
  
# Explore the top discharge dispositions
"""
SELECT discharge_disposition_id, COUNT(*) AS top_discharge_disposition
FROM encounter
GROUP BY discharge_disposition_id
ORDER BY top_discharge_disposition desc
LIMIT 10;

"""

### 1A.2 Identifying Missing or Anomalous Data

In [ ]:
#Check for missing values in the race column (Hint: Count the occurance of each unique value in this column)
"""
SELECT race AS p_race, COUNT(*) AS occurrence
FROM patient
GROUP BY race;

"""

#Check for missing or unusual values in the weight column
"""
SELECT
	CASE
		WHEN weight = '>200' THEN 'unusual'
		WHEN weight = '?' THEN 'missing'
		ELSE 'normal'
	END AS weight_status,
	COUNT(*) as occurrence
FROM patient
GROUP BY weight_status;

"""


### 1A.3 Understanding Age Distribution

In [ ]:
# Explore the age distribution of the patients
"""
SELECT age, COUNT(*) AS number_of_patients
FROM patient
GROUP BY age
ORDER BY number_of_patients DESC;

"""

### 1A.4 Admission Trends by Source and Type

In [ ]:
# Analyze how different admission sources contribute to hospital admissions 
# (Hint: Calculate the number of admissions per source)

"""
SELECT admission_source.description AS admission_source_description,
	COUNT(encounter.patient_nbr) AS number_of_patients
FROM encounter
JOIN admission_source ON encounter.admission_source_id = admission_source.admission_source_id
GROUP BY admission_source_description
ORDER BY number_of_patients DESC;

"""
#Investigate which admission types correspond to specific admission sources
# (Hint: add admission type to your previous query)
"""
SELECT admission_source.description AS admission_source,
       admission_type.description AS admission_type,
       COUNT(encounter.patient_nbr) AS number_of_patients
FROM encounter
JOIN admission_source ON encounter.admission_source_id = admission_source.admission_source_id
JOIN admission_type ON encounter.admission_type_id = admission_type.admission_type_id
GROUP BY admission_source.description, admission_type.description
ORDER BY number_of_patients DESC;

"""

### 1A.5 Hospital Stay and Readmission Patterns

In [ ]:
# Find the average time in hospital for each admission type
"""
SELECT admission_type.description AS admission_type,
       ROUND(AVG(encounter.time_in_hospital), 2) AS avg_time_in_hospital
FROM encounter
JOIN admission_type ON encounter.admission_type_id = admission_type.admission_type_id
GROUP BY admission_type.description
ORDER BY avg_time_in_hospital DESC;

"""
#Investigate readmission rates by admission type

"""
SELECT admission_type.description AS admission_type_description,
    COUNT(CASE WHEN encounter.readmitted = '<30' THEN 1 END) AS readmissions,
    ROUND((COUNT(CASE WHEN encounter.readmitted = '<30' THEN 1 END) * 100.0) / COUNT(*), 2) AS readmission_rate_percentage
FROM encounter
JOIN admission_type ON encounter.admission_type_id = admission_type.admission_type_id
GROUP BY admission_type.description
ORDER BY readmission_rate_percentage DESC;

"""

### 1A.6 Comparing Admission Types and Outcomes

In [ ]:
# Compare discharge dispositions across different admission types
"""
SELECT admission_type.description AS admission_type_description,
    discharge_disposition.description AS discharge_disposition_description,
    COUNT(*) AS number_of_patients
FROM encounter
JOIN admission_type ON encounter.admission_type_id = admission_type.admission_type_id
JOIN discharge_disposition ON encounter.discharge_disposition_id = discharge_disposition.discharge_disposition_id
GROUP BY admission_type.description, discharge_disposition.description
ORDER BY admission_type_description, number_of_patients DESC;

"""

# Compare readmission rates by discharge disposition
"""
SELECT discharge_disposition.description AS discharge_disposition_description,
    ROUND((COUNT(CASE WHEN encounter.readmitted = '<30' THEN 1 END) * 100.0) / COUNT(*), 2) AS readmission_rate_percentage
FROM encounter
JOIN discharge_disposition ON encounter.discharge_disposition_id = discharge_disposition.discharge_disposition_id
GROUP BY discharge_disposition.description
ORDER BY readmission_rate_percentage DESC;

"""

### 1B Exploratory Data Analysis with Python
### D escription
Perform comprehensive exploratory data analysis to understand the dataset's characteristics, patterns, and potential challenges. <br>
It is crucial to understand the structure and quality of the dataset before diving into any analysis or modeling.

### 1B.1: Loading and analysing the dataset shape:

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/diabetic_data.csv")

print("Dataset shape:", df.shape)
df.head()


### 1B.2: Load and Explore a Dataset Using NumPy

In [ ]:
import numpy as np

# Load the dataset
data = np.genfromtxt("../data/diabetic_data.csv", delimiter=',', dtype=None, encoding='utf-8', skip_header=1)

# Check the shape of the dataset
print("Dataset shape:", data.shape)

# Preview the first few rows
print("First 5 rows of the dataset:\n", data[:5])

### 1B.3: Analysing data types:

In [ ]:
import numpy as np

# Load the dataset
m_data = np.genfromtxt("../data/diabetic_data.csv", delimiter=',', skip_header=1, dtype=str)

# Check the data types of each column
print("Data types:", m_data.dtype)

# Preview unique values in specific categorical columns
categorical_columns = [1, 2, 3]

for col in categorical_columns:
    unique_values = np.unique(m_data[:, col])
    print(f"Unique values in column {col}:", unique_values)


### 1B.4: Exploratory Data Analysis (EDA) with Visualisations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data=df, x="admission_type_id", hue="age")

plt.show()

### 1B.5 Identify Metadata from the Dataset

In [ ]:
# metadata
import pandas as pd

df = pd.read_csv("../data/diabetic_data.csv")

print("Column Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nDataFrame Info:")
print(df.info())

print("\nSummary Statistics for Numerical Columns:")
print(df.describe())

print("\nSummary Statistics for Categorical Columns:")
print(df.describe(include=["object"]))

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicated Rows:")
print(df.duplicated().sum())


### 1B.6 Splitting Data for Analysis

In [ ]:
# split datase to train, test and validation set.
from sklearn.model_selection import train_test_split

X = df.drop(columns=['race','age'])
y = df['readmitted']

# Split into train and test sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

# Further split the training set into train and validation sets (80% of train for training, 20% of train for validation)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Testing set shape: {X_test.shape}")


## 2 Data Processing
### D escription
Create a robust data preprocessing pipeline to handle missing values, encode categorical variables, and scale numerical features.

### 2.1 Initial Cleaning and Pre-processing

#### 2.1.1: Load and Visualise the Data
- Use pandas to load the dataset.
- Visualise the first few rows to understand the structure of the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
df = pd.read_csv("../data/diabetic_data.csv")

# Set plot style
sns.set(style='whitegrid')

# 1. DiabetesMed Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='diabetesMed')
plt.title('Distribution of Diabetes Medication Use')
plt.xlabel('DiabetesMed')
plt.ylabel('Count')
plt.show()

# 2. Time in Hospital vs DiabetesMed (Boxplot)
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x='diabetesMed', y='time_in_hospital')
plt.title('Time in Hospital by Diabetes Medication Use')
plt.xlabel('DiabetesMed')
plt.ylabel('Time in Hospital (Days)')
plt.show()

# 3. Gender vs DiabetesMed
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='gender', hue='diabetesMed')
plt.title('Gender vs Diabetes Medication Use')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

# 4. DiabetesMed vs Readmission
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='diabetesMed', hue='readmitted')
plt.title('Diabetes Medication Use vs Readmission')
plt.xlabel('DiabetesMed')
plt.ylabel('Count')
plt.show()

# 5. Age Group vs DiabetesMed
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='age', hue='diabetesMed', order=sorted(df['age'].unique()))
plt.title('Age Group vs Diabetes Medication Use')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

# 6. Scatter: Num Medications vs Time in Hospital, hue by DiabetesMed
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='num_medications', y='time_in_hospital', hue='diabetesMed', alpha=0.6)
plt.title('Medications vs Time in Hospital')
plt.xlabel('Number of Medications')
plt.ylabel('Time in Hospital (Days)')
plt.show()

# 7. Scatter: Num Diagnoses vs Time in Hospital, hue by DiabetesMed
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='number_diagnoses', y='time_in_hospital', hue='diabetesMed', alpha=0.6)
plt.title('Diagnoses vs Time in Hospital')
plt.xlabel('Number of Diagnoses')
plt.ylabel('Time in Hospital (Days)')
plt.show()

# 8. Heatmap: Correlation Matrix of Numeric Features
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
corr_matrix = df[numeric_cols].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features')
plt.show()

# 9. DiabetesMed vs Inpatient Visits
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='diabetesMed', y='number_inpatient')
plt.title('Number of Inpatient Visits by Diabetes Medication Use')
plt.xlabel('DiabetesMed')
plt.ylabel('Number of Inpatient Visits')
plt.show()

# 10. Average Time in Hospital by Age Group
avg_time_age = df.groupby('age')['time_in_hospital'].mean().sort_index()
plt.figure(figsize=(10, 6))
avg_time_age.plot(kind='bar')
plt.title('Average Time in Hospital by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Time in Hospital (Days)')
plt.xticks(rotation=45)
plt.show()

#### 2.1.2: Display Data Types and Check for Incorrect Types

- Use the .dtypes attribute to display each column's data type.
- Identify columns that have unexpected data types (e.g., numeric data stored as strings).

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

# Load dataset
df = pd.read_csv("../data/diabetic_data.csv")

# List of categorical columns
categorical_columns = ['race', 'gender', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 
                       'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
                       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 
                       'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glyburide-metformin',
                       'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 
                       'metformin-pioglitazone', 'change']

# Convert categorical columns to category type
df[categorical_columns] = df[categorical_columns].astype('category')

# One-Hot Encoding for categorical columns
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=False)

# Convert boolean categorical columns to integers
df_encoded[df_encoded.select_dtypes(include='bool').columns] = df_encoded.select_dtypes(include='bool').astype(int)


# Convert 'age' to numeric values using the mapping
age_mapping = {
    '[0-10)': 5, 
    '[10-20)': 15, 
    '[20-30)': 25, 
    '[30-40)': 35, 
    '[40-50)': 45, 
    '[50-60)': 55, 
    '[60-70)': 65, 
    '[70-80)': 75, 
    '[80-90)': 85, 
    '[90-100)': 95
}

df_encoded['age'] = df_encoded['age'].map(age_mapping)

# Scaling the 'age' column with a MinMaxScaler after mapping it
scaler= MinMaxScaler()
scaler.fit(df_encoded[['age']])

df_encoded['Scaled age']= scaler.transform(df_encoded[['age']])

# Handle numerical columns and scaling
numerical_columns = ['num_lab_procedures', 'num_procedures', 'num_medications', 
                     'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'Scaled age']

# Scale numerical columns
scaler = StandardScaler()
df_encoded[numerical_columns] = scaler.fit_transform(df_encoded[numerical_columns])

# Drop identifiers and "weight" column
df_encoded = df_encoded.drop(columns=['encounter_id', 'patient_nbr', 'weight'])
 

In [ ]:
# Convert object to integers
for column in df_encoded.select_dtypes(include=['object']).columns:
    df_encoded[column] = df_encoded[column].astype('category').cat.codes
df_encoded = df_encoded.astype(int)
 
print(df_encoded.dtypes)

In [ ]:
# Define a function to convert values
def convert_1(value):
    if value == 'NO':
        return 0
    else:
        return 1

# Apply the function to the 'readmitted' column
df_encoded['readmitted'] = df['readmitted'].apply(convert_1)

In [ ]:
# Check the data types of each column
print("\nData types of each column:")
print(df_encoded.dtypes)

# Check the first few rows of the dataset
print("\nFirst few rows of the dataset:")
print(df_encoded.head())

# Check unique values in 'age' column
print("\nUnique values in 'age' column:", df_encoded['age'].unique())

# Check the scaling of numerical columns
print("\nSummary statistics for scaled numerical columns:")
print(df_encoded[numerical_columns].describe())


#### 2.1.3: Identify and Remove Duplicate Rows

- Check for duplicate rows.
- If duplicates exist, remove them 

In [ ]:
# Check for duplicate rows

duplicate_rows = df.duplicated()

print(f"Duplicate rows: {duplicate_rows.sum()}")

# Drop duplicates if any

clean_duplicate = df.drop_duplicates()

print(f"Number of rows after removing duplicates: {clean_duplicate.shape[0]}")

### 2.2 Handling Metadata and Missing Values
#### 2.2.1: Identify Metadata from the Dataset
Metadata includes information like column names, data types, and any additional descriptive information.

In [ ]:
# Display metadata including column names and data types

import pandas as pd

print("Column Names:")
print(df_encoded.columns.tolist())

print("\nData Types:")
print(df_encoded.dtypes)

print("\nDataFrame Info:")
print(df_encoded.info())

print("\nSummary Statistics for Numerical Columns:")
print(df_encoded.describe())

print("\nSummary Statistics for Categorical Columns:")
print(df_encoded.describe())

print("\nMissing Values:")
print(df_encoded.isnull().sum())

In [ ]:
# Count occurrences of '?' in every column
count_question_marks = (df_encoded == '?').sum()

# Filter to show only columns with '?' values
count_question_marks[count_question_marks > 0]

# Drop 'payer_code', 'medical_specialty' and (unscaled)'age'
df_encoded = df_encoded.drop(columns=['payer_code', 'medical_specialty', 'admission_type_id', 'discharge_disposition_id', 
                                      'admission_source_id', 'diag_1', 'diag_2', 'diag_3'])

# Replace '?' with NaN
df_encoded.replace('?', pd.NA, inplace=True)

# Check for missing values after replacement
df_encoded.isnull().sum()



#### 2.2.2: Analyse Different Types of Data
- Categorize columns based on their data types (e.g., numerical, categorical).
- Describe the significance of each data type and how it might impact data processing.

In [ ]:
# Describe data to understand different types
numerical_cols = df_encoded.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_encoded.select_dtypes(include=['object']).columns.tolist()

print("\nNumerical Columns:")
print(numerical_cols)

print("\nCategorical Columns:")
print(categorical_cols)

Numerical data is useful for mathematical operations, statistical analysis, and visualizations.



Categorical data is useful for grouping, segmentation, and encoding for machine learning models.

#### 2.2.3: Identify Missing Values
- Missing values can significantly impact model performance and analysis.
- Count missing values in each column.
- Calculate the percentage of missing values.

In [ ]:
# Identify and calculate the percentage of missing values
missing_percentage= (df_encoded.isnull().sum()/ len(df) * 100)
print(missing_percentage.round(3))

#### 2.2.4: Splitting Data for Analysis
Splitting data ensures that models are trained on one portion of the data and tested on another, unseen portion.

In [ ]:
from sklearn.model_selection import train_test_split

# Features and target
X = df_encoded.drop(columns=['time_in_hospital'])
y = df_encoded['time_in_hospital']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check the shapes of the splits
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

## 3 Machine Learning
### D escription
Implement various machine learning algorithms for regression, classification and clustering and evaluating them.

### 3.1 Implementing Regression Baseline
#### 3.1.1: Loading the data
- Use pandas to load the dataset.

#### 3.1.2: Start to implement Linear Regression as a baseline model

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
# Load the data
df = pd.read_csv("../data/diabetic_data.csv")

# Feature Selection
X = df_encoded.drop(columns=['time_in_hospital'])
y = df_encoded['time_in_hospital']

# Initialize the model
model = LinearRegression()

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)


#### 3.1.3: Visualise the output for linear regression

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot of predicted vs actual values
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, color='skyblue', alpha=0.6)

# Title and labels
plt.title('Predicted vs Actual Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')

# Plotting the line of perfect predictions (y = x)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='black')

plt.show()


#### 3.1.4: Implement Decision Tree Regression for Non-linear Relationships

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Feature Selection
X = df_encoded.drop(columns=['time_in_hospital'])
y = df_encoded['time_in_hospital']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Decision Tree Regressor model
dt_model = DecisionTreeRegressor(random_state=42)

# Train the model
dt_model.fit(X_train, y_train)

# Make predictions
dt_pred = dt_model.predict(X_test)

# Calculate MAE and MSE for Decision Tree model
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_mse = mean_squared_error(y_test, dt_pred)
dt_rmse = np.sqrt(dt_mse)

# The results (MAE and MSE)
print(f"Decision Tree MAE: {dt_mae}")
print(f"Decision Tree MSE: {dt_mse}")
print(f"Decision Tree RMSE: {dt_rmse}")

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot of Decision Tree predicted vs actual values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, dt_pred, color='mediumseagreen', alpha=0.6)

# Title and labels
plt.title('Decision Tree: Predicted vs Actual Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')

# Line of perfect predictions
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='black', linestyle='--')

plt.tight_layout()
plt.show()


#### 3.1.5: Implement Gradient Boosting Regression

In [ ]:
# Importing required sklearn packages for implementing Gradient Boosting Regressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
import pandas as pd

# Load the data
df = pd.read_csv("../data/diabetic_data.csv")

# Feature Selection
X = df_encoded.drop(columns=['time_in_hospital'])
y = df_encoded['time_in_hospital']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialising and training Gradient Boosting Regressor
gboost_model = GradientBoostingRegressor(random_state=42)

# Train the model
gboost_model.fit(X_train, y_train)

# Predictions
y_pred_gboost = gboost_model.predict(X_test)

# Evaluation
mae_gboost = mean_absolute_error(y_test, y_pred_gboost)
mse_gboost = mean_squared_error(y_test, y_pred_gboost)
rmse_gboost = np.sqrt(mse_gboost)

# Print results
print(f"Gradient Boosting Regression MAE: {mae_gboost}")
print(f"Gradient Boosting Regression MSE: {mse_gboost}")
print(f"Gradient Boosting Regression RMSE: {rmse_gboost}")

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot of Gradient Boosting predicted vs actual values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_gboost, color='orange', alpha=0.6)

# Title and labels
plt.title('Gradient Boosting: Predicted vs Actual Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')

# Line of perfect prediction
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='black', linestyle='--')

plt.tight_layout()
plt.show()

#### 3.1.6: Implement XGBoost Regression

In [ ]:
# Importing required packages for implementing XGBoost Regressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
import pandas as pd

# Load the data
df = pd.read_csv("../data/diabetic_data.csv")

# Feature Selection
X = df_encoded.drop(columns=['time_in_hospital'])
y = df_encoded['time_in_hospital']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initializing and training XGBoost Regressor
xgboost_model = XGBRegressor(random_state=42)

# Train the model
xgboost_model.fit(X_train, y_train)

# Predictions
y_pred_xgboost = xgboost_model.predict(X_test)

# Evaluation
mae_xgboost = mean_absolute_error(y_test, y_pred_xgboost)
mse_xgboost = mean_squared_error(y_test, y_pred_xgboost)
rmse_xgboost = np.sqrt(mse_xgboost)

# results
print(f"XGBoost Regression MAE: {mae_xgboost}")
print(f"XGBoost Regression MSE: {mse_xgboost}")
print(f"XGBoost Regression RMSE: {rmse_xgboost}")

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot of Gradient Boosting predicted vs actual values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_xgboost, color='tomato', alpha=0.6)

# Title and labels
plt.title('Gradient Boosting: Predicted vs Actual Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')

# Line of perfect prediction
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='black', linestyle='--')

plt.tight_layout()
plt.show()


#### 3.1.7: Visualise the output for non-linear regression

In [ ]:
## Write visualisation code here

# Customize colors and labels:
df_encoded['time_in_hospital'].hist(bins=20, color='skyblue', edgecolor='black', figsize=(10, 6))
plt.title('Histogram of Time in Hospital')
plt.xlabel('Time in Hospital')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

#### 3.1.8: Evaluation and Model Comparison
compare the performance of all four models: Linear Regression, Decision Tree Regression, Gradient Boosting, and XGBoost. Summarise each model’s performance using a comparison table.


In [ ]:
import pandas as pd
from tabulate import tabulate

results = {
    "Model": ["Linear Regression", "Decision Tree", "Gradient Boosting", "XGBoost"],
    "MAE": [mae, dt_mae, mae_gboost, mae_xgboost],
    "MSE": [mse, dt_mse, mse_gboost, mse_xgboost],  
    "RMSE": [rmse, dt_rmse, rmse_gboost, rmse_xgboost]
}

results_df = pd.DataFrame(results)

print(tabulate(results_df, headers= "keys", tablefmt="grid"))


### 3.2 Implementing Classification Baseline


#### 3.2.1: Loading the data
- Use pandas to load the dataset.

In [ ]:
## Loading the data
import pandas as pd

df= pd.read_csv("../data/diabetic_data.csv")

#### 3.2.2:  Prepare Data for Classification


In [ ]:
# Define a function to convert values
def convert_2(value):
    if value == 'No':
        return 0
    else:
        return 1

# Apply the function to the 'diabetesMed' column
df_encoded['diabetesMed'] = df['diabetesMed'].apply(convert_2)

#### 3.2.3:  Implement Logistic Regression as a Baseline Model

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load your dataset
df = pd.read_csv("../data/diabetic_data.csv")

# Select features for X
selected_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'
]

X = df_encoded[selected_features]
y = df_encoded['diabetesMed']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Logistic Regression model
model_lg = LogisticRegression(C=0.01, max_iter=100, solver='liblinear', random_state=42)

# Train the model on the training data
model_lg.fit(X_train, y_train)

# Predictions and evaluation
y_pred_lg = model_lg.predict(X_test)

# Evaluation metrics
lg_accuracy = accuracy_score(y_test, y_pred_lg)
lg_precision = precision_score(y_test, y_pred_lg)
lg_recall = recall_score(y_test, y_pred_lg)
lg_f1 = f1_score(y_test, y_pred_lg)

print("Logistic Regression Accuracy:", round(lg_accuracy, 3))
print("Logistic Regression Precision:", round(lg_precision, 3))
print("Logistic Regression Recall:", round(lg_recall, 3))
print("Logistic Regression F1-Score:", round(lg_f1, 3))

#### 3.2.4: Implement Random Forest Classifier as a Non-linear Baseline

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load your dataset
df = pd.read_csv("../data/diabetic_data.csv")

# Define selected features
selected_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'
]

# Prepare X and y
X = df_encoded[selected_features]
y = df_encoded['diabetesMed']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# Train the model
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

# Evaluate
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

# Output the results
print("Random Forest Accuracy:", round(rf_accuracy, 3))
print("Random Forest Precision:", round(rf_precision, 3))
print("Random Forest Recall:", round(rf_recall, 3))
print("Random Forest F1-Score:", round(rf_f1, 3))

#### 3.2.5: Implement K-Nearest Neighbors (KNN) Classifier

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load your dataset
df = pd.read_csv("../data/diabetic_data.csv")

# Define selected features
selected_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'
]

# Prepare X and y
X = df_encoded[selected_features]
y = df_encoded['diabetesMed']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the KNN model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Make predictions
y_pred_knn = knn.predict(X_test)

# Evaluate the model
knn_accuracy = accuracy_score(y_test, y_pred_knn)
knn_precision = precision_score(y_test, y_pred_knn)
knn_recall = recall_score(y_test, y_pred_knn)
knn_f1 = f1_score(y_test, y_pred_knn)

# Print the evaluation results
print("KNN Accuracy:", round(knn_accuracy, 3))
print("KNN Precision:", round(knn_precision, 3))
print("KNN Recall:", round(knn_recall, 3))
print("KNN F1-Score:", round(knn_f1, 3))


#### 3.2.6: Evaluation and Model Comparison
Compare the performance of implemented models using four metrics including "Accuracy", "Precision", "Recall", and "F1-score".

In [ ]:
import pandas as pd
from tabulate import tabulate

# Number of decimal places to round to
decimal_places = 3

# Round each metric in the results dictionary
results = {
    "Model": ["Logistic Regression", "Random Forest", "K-Nearest Neighbors"],
    "Accuracy": [round(val, decimal_places) for val in [lg_accuracy, rf_accuracy, knn_accuracy]],
    "Precision": [round(val, decimal_places) for val in [lg_precision, rf_precision, knn_precision]],
    "Recall": [round(val, decimal_places) for val in [lg_recall, rf_recall, knn_recall]],
    "F1-Score": [round(val, decimal_places) for val in [lg_f1, rf_f1, knn_f1]]
}


results_df = pd.DataFrame(results)

print(tabulate(results_df, headers= "keys", tablefmt="grid"))


Logistic Regression:
Logistic Regression is simple, interpretable, and efficient for linearly separable data but struggles with complex, non-linear relationships.

Random Forest:
Random Forest is robust, handles non-linear relationships well, and is less prone to overfitting, but it can be computationally expensive and harder to interpret.

K-Nearest Neighbors (KNN):
KNN is flexible and handles complex patterns but is sensitive to the choice of k and can be slow with large datasets, requiring feature normalization for better performance.

### 3.3 Implementing Clustering Baseline

#### 3.3.1: Load and Preprocess the Data

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load the dataset

# Standardising column names and remove duplicates
df_encoded.columns = df_encoded.columns.str.replace('>', '_gt_')
df_encoded.columns = df_encoded.columns.str.replace('?', '_unknown')
df_encoded.columns = df_encoded.columns.str.replace('/', '_')

# Applying standard scaling to numerical features for clustering
numeric_columns = df_encoded.select_dtypes(include=['float64', 'int64']).columns
scaler = StandardScaler()
df_encoded[numeric_columns] = scaler.fit_transform(df_encoded[numeric_columns])

column_names = df_encoded.columns.tolist()
print(column_names)


#### 3.3.2: Implement K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# Features
X = df_encoded[['Scaled age', 'num_lab_procedures', 'num_medications', 'time_in_hospital', 'readmitted']]

# Elbow method to find the optimal K 
inertia = []
K_range = range(1, 11) 

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

# Plot the Elbow curve
plt.plot(K_range, inertia, marker='o')
plt.title('Elbow Method For Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Sum of Squared Distances)')
plt.show()

In [ ]:
optimal_K = 3

# Initialising and training K-Means model
kmeans = KMeans(n_clusters= optimal_K, random_state=42)
kmeans.fit(X)

# Predict clusters and evaluate
silhouette_avg = silhouette_score(X, kmeans.labels_)
print(f"K-Means Silhouette Score: {silhouette_avg}")

#### 3.3.3: Apply PCA for Dimensionality Reduction

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from mpl_toolkits.mplot3d import Axes3D

# Applying PCA to reduce dimensions
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Train KMeans clustering on PCA-reduced data
kmeans = KMeans(n_clusters= optimal_K, random_state=42)
kmeans.fit(X_pca)

# Visualising clusters for K-Means in 2D
# Hint: Set n_components=2 for 2D visualisation or n_components=3 for 3D visualisation.
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans.labels_, cmap='viridis', s=50, alpha=0.7)
plt.title('K-Means Clustering (PCA-reduced Data)', fontsize=16)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(label='Cluster Label')
plt.show()

# Silhouette Score for the 2D PCA-reduced data
silhouette_2d = silhouette_score(X_pca, kmeans.labels_)
print(f"Silhouette Score for K-Means on 2D PCA Data: {silhouette_2d}")

optimal_k = 3

# Apply PCA for 3D visualization
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X)

# Train K-Means on the 3D data
kmeans_3d = KMeans(n_clusters=optimal_k, random_state=42)
kmeans_3d.fit(X_pca_3d)

# Plot the 3D scatter plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_pca_3d[:, 0], X_pca_3d[:, 1], X_pca_3d[:, 2], c=kmeans_3d.labels_, cmap='viridis', s=50, alpha=0.7)
ax.set_title('K-Means Clustering on PCA-Reduced Data (3D)', fontsize=16)
ax.set_xlabel('Principal Component 1', fontsize=12)
ax.set_ylabel('Principal Component 2', fontsize=12)
ax.set_zlabel('Principal Component 3', fontsize=12)
plt.show()

# Silhouette Score for the 3D PCA-reduced data
silhouette_3d = silhouette_score(X_pca_3d, kmeans_3d.labels_)
print(f"Silhouette Score for K-Means on 3D PCA Data: {silhouette_3d}")

#### 3.3.4: Implement Random Hierarchical Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Import resample package to solve memory issues
from sklearn.utils import resample

# Subsampling the dataset to 10k samples
X_sampled = resample(X, n_samples=10000, random_state=42)

# Initialising and fitting hierarchical clustering
hierarchical = AgglomerativeClustering(n_clusters=optimal_K, linkage='ward')
hierarchical_labels = hierarchical.fit_predict(X_sampled)

# Evaluating with Silhouette Score
hierarchical_silhouette = silhouette_score(X_sampled, hierarchical_labels) 
print(f"Hierarchical Clustering Silhouette Score: {hierarchical_silhouette}")

#### 3.3.5: Implement DBSCAN for Density-Based Clustering

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import numpy as np

# Set parameters for DBSCAN
eps = 0.5
min_samples = 10

# Apply PCA
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X)

# Initialising and fiting DBSCAN
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(X_reduced)

# Filtering noise points (labeled as -1)
core_samples_mask = dbscan_labels != -1
X_core = X_reduced[core_samples_mask]
dbscan_core_labels = dbscan_labels[core_samples_mask]

# Evaluate with Silhouette Score (excluding noise)
if len(np.unique(dbscan_core_labels)) > 1:
    dbscan_silhouette = silhouette_score(X_core, dbscan_core_labels)
    print(f"Silhouette Score for DBSCAN (excluding noise): {dbscan_silhouette}")

#### 3.3.6: Evaluation and Model Comparison

In [ ]:
from tabulate import tabulate

# Summary of Silhouette Scores
results = {
    "Clustering Method": ["K-Means", "Hierarchical", "DBSCAN"],
    "Silhouette Score": [silhouette_2d, hierarchical_silhouette, dbscan_silhouette]
}

results_df = pd.DataFrame(results)

print(tabulate(results_df, headers= "keys", tablefmt="grid"))

### 3.4 Feature Engineering and Feature Selection
Select the most relevant features to improve model performance

#### 3.4.1: Feature Engineering

In [ ]:
# Create age group categories if applicable
age_mapping = {
    '[0-10)': 5, 
    '[10-20)': 15, 
    '[20-30)': 25, 
    '[30-40)': 35, 
    '[40-50)': 45, 
    '[50-60)': 55, 
    '[60-70)': 65, 
    '[70-80)': 75, 
    '[80-90)': 85, 
    '[90-100)': 95
}

df['age'] = df['age'].map(age_mapping)

# Interaction feature example

df['age_hospital_interaction'] = df['age'] * df['time_in_hospital']

# Verify the interaction feature
print(df[['age', 'time_in_hospital', 'age_hospital_interaction']].head())


#### 3.4.2: Generate Statistical Features
Calculate aggregate statistics such as mean, max, min, or standard deviation for features if relevant

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Calculate statistical features for 'age'
df['age_mean'] = df['age'].mean()
df['age_max'] = df['age'].max()
df['age_min'] = df['age'].min()
df['age_std'] = df['age'].std()

# Verify the new features
print(df[['age', 'age_mean', 'age_max', 'age_min', 'age_std']].head())


#### 3.4.3: Feature Transformation
Normalise or standardise features, especially for distance-based models like KNN or clustering

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Standardize numerical columns
numerical_columns = ['num_lab_procedures', 'num_procedures', 'num_medications', 
                     'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age']

# Scale numerical columns
scaler = StandardScaler()
df[numerical_columns] = scaler.fit_transform(df[numerical_columns])


#### 3.4.4: Feature Selection
Use feature importance from tree-based models (Random Forest, Gradient Boosting) to select important features


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical target as numeric
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df_encoded['diabetesMed'])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

# Define keywords to exclude (medications + insulin)
diabetes_related_keywords = [
    'insulin', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]

# Filter out diabetes-related features from X
X_filtered = df_encoded.drop(columns=['diabetesMed'])  # initial drop of target
filtered_columns = [col for col in X_filtered.columns if not any(drug in col for drug in diabetes_related_keywords)]
X_filtered = X_filtered[filtered_columns]

# Target variable
y = y_encoded

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_filtered, y, test_size=0.2, random_state=42)

# Train Random Forest
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Feature importances
importances = model.feature_importances_
feature_names = X_filtered.columns
indices = np.argsort(importances)[::-1]

# Top 10 features (non-diabetes-related)
top_10_indices = indices[:10]
top_10_importances = importances[top_10_indices]
top_10_feature_names = feature_names[top_10_indices]

# Plot top 10
plt.figure(figsize=(10, 6))
plt.title("Top 10 Feature Importances (Excluding Diabetes-Related Features)")
plt.bar(range(len(top_10_indices)), top_10_importances, align="center")
plt.xticks(range(len(top_10_indices)), top_10_feature_names, rotation=90)
plt.xlabel("Feature")
plt.ylabel("Importance Score")
plt.tight_layout()
plt.show()

#### 3.4.5: Model Evaluation After Feature Selection
- After feature engineering and selection, evaluate your model with the new set of features.
- Apply cross-validation to measure the model’s performance with the refined feature set

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Use the top non-diabetes-related features (from your earlier feature importance plot)
X_selected = df_encoded[[
    'change_Ch', 'change_No', 'time_in_hospital', 'age', 'num_lab_procedures',
    'num_medications', 'number_inpatient', 'number_diagnoses',
    'number_outpatient', 'num_procedures'
]]

y = y_encoded  # Binary classification target: diabetesMed

# Initialize Logistic Regression (as a classifier)
model = LogisticRegression(random_state=42, max_iter=1000)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(model, X_selected, y, cv=5, scoring='accuracy')

# Output the average accuracy
print("Cross-validated accuracy (Logistic Regression):", round(cv_scores.mean(), 4))

### 3.5 Hyperparameter Tuning
Improve the quality of your implementations


#### 3.5.1: GridSearchCV for Systematic Hyperparameter Tuning
- Define Hyperparameter Grid for Different Models
- Define grids of hyperparameters for models such as Logistic Regression (for classification) and K-Means (for clustering).

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans

# Define hyperparameter grid for Logistic Regression
logistic_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [500, 700, 1000]  
}

# Initialize the logistic regression model
logistic_model = LogisticRegression(random_state=42)

# Perform GridSearchCV for Logistic Regression
logistic_search = GridSearchCV(estimator=logistic_model, param_grid=logistic_grid, cv=5, scoring='accuracy')
logistic_search.fit(X_selected, y_encoded)

print("Best parameters for Logistic Regression:", logistic_search.best_params_)
print("Best cross-validated accuracy:", logistic_search.best_score_)

# Define hyperparameter grid for KMeans
kmeans_grid = {
    'n_clusters': [2, 3, 4, 5, 6],
    'init': ['k-means++', 'random'],
    'max_iter': [500, 600, 700],
    'n_init': [10, 20, 30]  
}

# Initialize the KMeans model
kmeans_model = KMeans(random_state=42)

# Removed y_encoded from fit()
kmeans_search = GridSearchCV(estimator=kmeans_model, param_grid=kmeans_grid, cv=5)
kmeans_search.fit(X_selected)

print("Best parameters for KMeans:", kmeans_search.best_params_)
print("Best clustering score (from KMeans.score):", kmeans_search.best_score_)

#### 3.5.2 Apply GridSearchCV with Cross-Validation
- Use GridSearchCV with cross-validation to perform hyperparameter tuning on both models.

In [ ]:
# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) 

# KMeans GridSearchCV
kmeans_search = GridSearchCV(KMeans(), kmeans_grid, scoring='adjusted_rand_score', cv=3, n_jobs=-1)
kmeans_search.fit(X_train_scaled, y_train)

In [ ]:
# Logistic Regression GridSearchCV
logistic_search = GridSearchCV(LogisticRegression(), logistic_grid, scoring='accuracy', cv=5, n_jobs=-1)
logistic_search.fit(X_train_scaled, y_train)

#### 3.5.3: Nested Cross-Validation to Validate Stability of Hyperparameters

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold

# Define logistic regression model and parameter grid
logistic = LogisticRegression(max_iter=1000, random_state=42)
param_grid = {'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear', 'lbfgs']}

# Outer cross-validation loop
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Inner cross-validation loop for hyperparameter tuning
logistic_search = GridSearchCV(estimator=logistic, param_grid=param_grid, scoring='accuracy', cv=3)

# Nested cross-validation
nested_cv_score_logistic = cross_val_score(logistic_search, X_train_scaled, y_train, cv=outer_cv, scoring='accuracy')

# Print nested CV accuracy
print("Nested CV Accuracy for Logistic Regression:", nested_cv_score_logistic.mean())

#### 3.5.4: Evaluate Tuned Models with Multiple Metrics
- Evaluate the tuned Logistic Regression model on both training and test sets.
- Use multiple metrics for classification (accuracy, F1-score) and clustering (silhouette score, Calinski-Harabasz).

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, silhouette_score, calinski_harabasz_score

# Fit GridSearchCV objects
logistic_search.fit(X_train_scaled, y_train)
kmeans_search.fit(X_train_scaled)

# Logistic Regression evaluation
y_train_pred = logistic_search.best_estimator_.predict(X_train_scaled)
y_test_pred = logistic_search.best_estimator_.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted')
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"Logistic Regression - Train Accuracy: {train_accuracy}")
print(f"Logistic Regression - Test Accuracy: {test_accuracy}")
print(f"Logistic Regression - Train F1: {train_f1}")
print(f"Logistic Regression - Test F1: {test_f1}")

# KMeans evaluation
kmeans_labels = kmeans_search.best_estimator_.predict(X_train_scaled)
silhouette = silhouette_score(X_train_scaled, kmeans_labels)
calinski_harabasz = calinski_harabasz_score(X_train_scaled, kmeans_labels)

print(f"KMeans - Silhouette Score: {silhouette}")
print(f"KMeans - Calinski-Harabasz Score: {calinski_harabasz}")

### 3.6 Model Selection
- Compare and evaluate the implemented models and select the best ones.

#### 3.6.1: Compare and Choose the Best Model
- Compare the models using evaluation metrics (e.g., MSE for regression, F1-score for classification, Silhouette Score for clustering).
- Summarise the performance metrics of each model in a table and choose the model with the best performance for each task.

In [ ]:
import pandas as pd
from tabulate import tabulate

# Define each task and relevant models/metrics
results = {
    "Task": ["Regression", "Regression", "Classification", "Clustering"],
    "Model": ["XGBoost", "Gradient Boosting", "K-Nearest Neighbors", "K-Means"],
    "Best Metric": ["RMSE", "MAE", "F1-Score", "Silhouette Score"],
    "Metric Value": [rmse_xgboost, mae_gboost, knn_f1, silhouette_2d]
}

# Create DataFrame
results_df = pd.DataFrame(results)

# Print in a nice table format
print(tabulate(results_df, headers="keys", tablefmt="grid"))

# Model Selection Analysis

This section summarizes and explains the rationale behind selecting the best model for each machine learning task (Regression, Classification, Clustering) based on their performance metrics.

---

## Regression Task – XGBoost Regressor
- **Selected Model:** `XGBoost Regressor`
- **Best Metric:** Root Mean Squared Error (RMSE) = **2.548**
- **Justification:**
  - Achieved the **lowest RMSE and MAE** among all tested regression models (Linear Regression, Decision Tree, Gradient Boosting, XGBoost).
  - Outperformed Gradient Boosting slightly, showing better bias-variance trade-off.
  - XGBoost handles **non-linear patterns**, **missing values**, and **regularization** effectively, which is ideal for structured healthcare data like this.

---

## Classification Task – K-Nearest Neighbors (KNN)
- **Selected Model:** `K-Nearest Neighbors`
- **Best Metric:** F1-Score = **0.836**
- **Justification:**
  - Achieved the **highest F1-score**, indicating a strong balance between precision and recall.
  - Outperformed:
    - **Logistic Regression**, which had perfect recall (1.0) but lower precision.
    - **Random Forest**, which had good precision but lower recall and overall F1.
  - KNN’s effectiveness suggests that **proximity-based decision boundaries** work well for this classification problem.

---

## Clustering Task – K-Means
- **Selected Model:** `K-Means Clustering`
- **Best Metric:** Silhouette Score = **0.458**
- **Justification:**
  - Highest Silhouette Score among clustering methods:
    - K-Means: **0.458**
    - Hierarchical: 0.384
    - DBSCAN: 0.066
  - Indicates that K-Means produces the most cohesive and well-separated clusters in this dataset.
  - Suitable for structured, moderately scaled healthcare data.

---

## Summary Table

| Task          | Best Model           | Metric           | Value   | Reasoning                                         |
|---------------|----------------------|------------------|---------|--------------------------------------------------|
| Regression    | XGBoost              | RMSE             | 2.548   | Most accurate and robust on numeric features     |
| Classification| K-Nearest Neighbors  | F1-Score         | 0.836   | Best balance of precision and recall             |
| Clustering    | K-Means              | Silhouette Score | 0.458   | Most cohesive and distinct clusters              |

---

#### 3.6.2: Finalise and Save the Best Model
- Once the best model is identified, retrain it on the full dataset and save it using joblib for future use.

In [ ]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

# Save the best regression model (XGBoost)
joblib.dump(xgboost_model, '../models/best_regression_model_xgboost.pkl')

# Save the best classification model (KNN)
joblib.dump(knn, '../models/best_classification_model_knn.pkl')

# Save the clustering model (K-Means)
joblib.dump(kmeans, '../models/best_clustering_model_kmeans.pkl')

## 4 Mathematics for Machine Learning

Linear and logistic regression are implemented from scratch with NumPy in `02_linear_regression_from_scratch.ipynb` and `03_logistic_regression_from_scratch.ipynb`.